In [ ]:
%pip install pixeltable-pgserver
%pip install psycopg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pixeltable-pgserver]


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
import pixeltable_pgserver as pgserver
server = pgserver.get_server('/tmp/my_db_data')
db_url = server.get_uri()
print("Посилання для %sql:", db_url)

engine = create_engine(server.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


Посилання для %sql: postgresql://postgres:@/postgres?host=/tmp/my_db_data
PostgreSQL 18.4 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [ ]:
%load_ext sql
%sql postgresql://postgres:@/postgres?host=/tmp/my_db_data
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


# Завдання 1. Setup і typed-схема Olist

In [ ]:
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [ ]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [ ]:
%%sql
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


[]

In [ ]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
99441 rows affected.
99441 rows affected.
32951 rows affected.
3095 rows affected.
112650 rows affected.


[]

In [ ]:
%%sql
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
5 rows affected.


table_name,n_rows
customers,99441
order_items,112650
orders,99441
products,32951
sellers,3095


Кількість рядків в усіх таблицях однакова. Таблиці відсортовані по назві в алфавітному порядку.

# Завдання 2. CTE-chain pipeline з ≥5 stages

In [ ]:
%%sql
WITH
reference_date AS (
    -- Stage 0: Стабільна опорна дата для історичного датасету
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: Замовлення + фінансові показники + логістичні метрики
    -- Прив'язуємо customer_unique_id одразу, щоб рахувати метрики на рівні людини
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        -- Обчислюємо затримку доставки (додатні значення — запізнення, від'ємні — приїхало раніше терміну)
        EXTRACT(DAY FROM (o.order_delivered_customer_date - o.order_estimated_delivery_date))::INTEGER AS delivery_delay_days,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total,
        -- Перевіряємо, чи є в замовленні бодай один продавець з іншого штату
        MAX(CASE WHEN s.seller_state <> c.customer_state THEN 1 ELSE 0 END) AS is_cross_state
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON o.customer_id = c.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_sellers AS s
        ON oi.seller_id = s.seller_id
    GROUP BY
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
),
stage2_rfm AS (
    -- Stage 2: Справжні RFM метрики клієнта + міжпокупковий інтервал
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(DISTINCT s.order_id) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date,
        -- Середня затримка доставки для клієнта
        ROUND(AVG(COALESCE(s.delivery_delay_days, 0)), 1)::NUMERIC(6, 1) AS avg_delivery_delay_days,
        -- Частка замовлень, відправлених з інших штатів
        ROUND(AVG(s.is_cross_state), 2)::NUMERIC(3, 2) AS cross_state_ratio,
        -- Середній інтервал між покупками (для клієнтів із > 1 замовленням)
        CASE
            WHEN COUNT(DISTINCT s.order_id) > 1
            THEN ROUND((MAX(s.order_date) - MIN(s.order_date))::NUMERIC / (COUNT(DISTINCT s.order_id) - 1), 1)
            ELSE NULL
        END AS avg_interpurchase_days
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_unique_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: Квартальна когорта на основі першого замовлення клієнта
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: Скользящий 30-денний AOV на рівні унікального користувача
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: Коефіцієнт скасувань на рівні унікального клієнта
    SELECT
        customer_unique_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_unique_id
),
stage6_category_revenue AS (
    -- Stage 6: Продажі за категоріями та кількість унікальних товарів
    SELECT
        c.customer_unique_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue,
        COUNT(DISTINCT oi.product_id) AS unique_products_count
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON o.customer_id = c.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY c.customer_unique_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: Визначення улюбленої категорії та диверсифікації кошика
    SELECT
        customer_unique_id,
        product_category_name,
        category_revenue,
        total_unique_products
    FROM (
        SELECT
            cr.*,
            SUM(unique_products_count) OVER (PARTITION BY cr.customer_unique_id) AS total_unique_products,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_unique_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
-- Фінальна збірка розширеної вітрини ознак
SELECT
    ca.customer_unique_id,
    -- Фіксуємо базову географію через останній відомий стан клієнта
    (SELECT customer_state FROM olist_customers WHERE customer_unique_id = ca.customer_unique_id LIMIT 1) AS customer_state,

    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,

    -- Нові логістичні ознаки
    coh.avg_delivery_delay_days,
    coh.cross_state_ratio,
    coh.avg_interpurchase_days,

    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,

    -- Нові продуктові ознаки
    COALESCE(pc.total_unique_products, 0) AS total_unique_products_bought,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM (SELECT DISTINCT customer_unique_id FROM olist_customers)  AS ca
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_unique_id = ca.customer_unique_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_unique_id = ca.customer_unique_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_unique_id = ca.customer_unique_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_unique_id = ca.customer_unique_id
ORDER BY monetary DESC NULLS LAST, ca.customer_unique_id
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
20 rows affected.


customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_delivery_delay_days,cross_state_ratio,avg_interpurchase_days,avg_order_value_30d_recent,cancellation_rate,total_unique_products_bought,preferred_category,preferred_category_revenue
0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,-5.0,1.00,None,13664.08,0.0000,1,telefonia_fixa,13664.08
da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,-17.5,1.00,0.0,3785.82,0.0000,2,eletroportateis,7571.63
763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,-6.0,1.00,None,7274.88,0.0000,1,telefonia_fixa,7274.88
dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,-5.0,1.00,None,6929.31,0.0000,1,utilidades_domesticas,6929.31
459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,5.0,1.00,None,6922.21,0.0000,1,pcs,6922.21
ff4159b92c40ebe40454e3e6a7c35ed6,SP,512,1,6726.66,2017-05-24,2017-04-01,-22.0,1.00,None,6726.66,0.0000,1,artes,6726.66
4007669dec559734d6f53e029e360987,MG,328,1,6081.54,2017-11-24,2017-10-01,-8.0,1.00,None,6081.54,0.0000,1,agro_industria_e_comercio,6081.54
5d0a2980b292d049061542014e8960bf,GO,98,1,4809.44,2018-07-12,2018-07-01,0.0,1.00,None,4809.44,1.0000,1,cool_stuff,4809.44
eebb5dda148d3893cdaf5b5ca3040ccb,SP,548,1,4764.34,2017-04-18,2017-04-01,-7.0,0.00,None,4764.34,0.0000,1,eletroportateis,4764.34
48e1ac109decbb87765a3eade6854098,PB,118,1,4681.78,2018-06-22,2018-04-01,-10.0,1.00,None,4681.78,0.0000,1,pcs,4681.78


# Завдання 3. Recursive CTE для referral chain

In [ ]:
%%sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
Done.
9 rows affected.
9 rows affected.


referrer_customer_id,referred_customer_id,referred_at
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-25 11:25:59.414600+00:00
000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-25 11:25:59.414600+00:00
0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-25 11:25:59.414600+00:00
0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-25 11:25:59.414600+00:00
000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-25 11:25:59.414600+00:00
0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-25 11:25:59.414600+00:00
000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-25 11:25:59.414600+00:00
00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-25 11:25:59.414600+00:00
00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-25 11:25:59.414600+00:00


In [ ]:
%%sql
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
10 rows affected.


depth,n_customers,sample_path
1,99432,ffffe8b65bbe3087b653a978c870db99
2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


При виникненні циклічної залежності в даних (наприклад, коли користувач A запросив користувача B, а той у свою чергу «запросив» назад користувача A: A → B → A), рекурсивний SQL-запит без спеціального захисту потрапить у нескінченний цикл.
1. База даних буде нескінченно виконувати Recursive member (JOIN).
2. Рядок для користувача A буде додаватися на парних рівнях глибини, а для B — на непарних.
3. Показник depth зростатиме до нескінченності.
4. Текстовий рядок path буде розростатися (A -> B -> A -> B -> A...), поки не викличе переповнення пам'яті (Out of Memory) або вичерпання дискового простору під тимчасові таблиці (temp files), після чого транзакція аварійно завершиться з помилкою.


# Завдання 4. Materialized View з CONCURRENT REFRESH

In [ ]:
%%sql
-- 1. Створення Materialized View з префіксом olist_
CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

-- 2. Створення унікального індексу на нову назву представлення
CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
ON mv_olist_customer_features (customer_id);

-- 3. Первинне тестування конкурентного оновлення
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
99441 rows affected.
Done.
Done.


[]

Перевіряємо працездатність MV:

In [ ]:
%%sql
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;

SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
1 rows affected.
5 rows affected.


customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


In [ ]:
%%sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.


[]

# Завдання 5. EXPLAIN ANALYZE

Сценарій 1: Повне переобчислення через сирий CTE-пайплайн

In [ ]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 5;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
136 rows affected.


QUERY PLAN
Limit (cost=118581.45..118581.46 rows=5 width=150) (actual time=4398.870..4398.897 rows=5.00 loops=1)
"Buffers: shared hit=111255, temp read=9185 written=9334"
CTE stage1_orders_total
-> GroupAggregate (cost=0.83..27069.78 rows=99441 width=103) (actual time=0.056..443.784 rows=99441.00 loops=1)
Group Key: o_1.order_id
Buffers: shared hit=103453
-> Merge Left Join (cost=0.83..24484.69 rows=112650 width=95) (actual time=0.034..267.364 rows=113425.00 loops=1)
Merge Cond: (o_1.order_id = oi_1.order_id)
Buffers: shared hit=103453
-> Index Scan using olist_orders_pkey on olist_orders o_1 (cost=0.42..12563.67 rows=99441 width=83) (actual time=0.018..93.201 rows=99441.00 loops=1)



Загальний час виконання (Execution Time): 4398.89 мс (~4.4 секунди) для отримання всього 5 рядків (LIMIT 5). • Дискова підсистема (Buffers): Пайплайн виконав величезний обсяг роботи в оперативній пам'яті та на диску:

* shared hit=111255 — прочитано ~111 тисяч сторінок із кешу (близько 870 МБ даних).
* temp read=9185 written=9334 — база даних скинула на диск ~73 МБ тимчасових файлів для виконання сортувань, оскільки вони не помістилися у work_mem.


Аналіз найдорожчих вузлів пайплайну:
Нижче наведено три найпроблемніші місця («вузькі горлечка»), які гальмують усю роботу:  
1. **Сортування через диск (етап stage6_category_revenue)**   
База даних намагалася відсортувати виручку за категоріями. Обсяг даних виявився завеликим, і вони не помістилися в оперативну пам'ять. Базі довелося скинути ~8 МБ тимчасових даних на диск, щоб продовжити сортування там. Це забрало майже 1 секунду і сильно уповільнило процес.   
2. **Важкі розрахунки для рейтингу категорій (етап stage7)**
База визначає «улюблену категорію» для кожного клієнта, присвоюючи їм порядкові номери. Хоча саме сортування тут пройшло відносно вдало, сам інструмент підрахунку рейтингу (WindowAgg) змушений обробляти просто гігантський масив рядків. Через це ланцюжок операцій забирає найбільше часу в одному конкретному місці — близько 1,4 секунди.   
3. **Перевантаження пам'яті через об'єднання таблиць (Каскад Hash Join)**
Пайплайн побудований так, що спочатку окремо рахуються різні показники (когорти, середній чек, скасування), а потім усе це склеюється в одну велику таблицю. Оскільки кожен проміжний етап створює величезну купу даних, Postgres не може втримати їх у пам'яті під час об'єднання. Йому доводиться створювати величезні тимчасові файли на диску (записи temp written). В результаті фінальне склеювання перетворюється на «снігову кулю» і розтягується аж до 4,3 секунди.




In [ ]:
Сценарій 2: Lookup безпосередньо з Materialized View

In [ ]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
16 rows affected.


QUERY PLAN
Nested Loop (cost=0.83..8.91 rows=1 width=92) (actual time=0.040..0.042 rows=1.00 loops=1)
Buffers: shared hit=8
-> Limit (cost=0.42..0.47 rows=1 width=33) (actual time=0.021..0.022 rows=1.00 loops=1)
Buffers: shared hit=4
-> Index Only Scan using olist_customers_pkey on olist_customers (cost=0.42..5328.03 rows=99441 width=33) (actual time=0.020..0.020 rows=1.00 loops=1)
Heap Fetches: 0
Index Searches: 1
Buffers: shared hit=4
-> Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv (cost=0.42..8.44 rows=1 width=92) (actual time=0.014..0.014 rows=1.00 loops=1)
Index Cond: (customer_id = olist_customers.customer_id)



Заміна живого розрахунку (CTE) на заздалегідь збережений результат (Materialized View) перетворила важку аналітичну роботу бази даних на миттєвий пошук інформації. Це як шукати слово в словнику замість того, щоб щоразу перечитувати всю енциклопедію.
Давайте розберемо три головні перемоги цієї оптимізації:

1. Миттєвий пошук через «індексний довідник» (Index Scan)

Коли нам були потрібні були дані по клієнту, база даних за допомогою CTE заново з нуля вираховувала складні формули (когорти, рейтинги, історію покупок).
Оскільки ми в MV створили унікальний індекс за customer_id, база більше нічого не рахує. Вона дивиться в алфавітний покажчик (індекс) і одразу знаходить потрібний рядок. Замість лопатити 870 МБ даних в пам'яті, вона відкриває всього 4 маленькі блоки (32 КБ).

2. Висока швидкість (0.091 мс)

При виконанні CTE запит думав майже 4.4 секунди. При використанні MV база впоралася менш ніж за одну десяту мілісекунди (це в 48 000 разів швидше!). Це стало можливим, тому що всі складні розрахунки вже «заморожені» (матеріалізовані) і просто лежать на диску у вигляді готової таблиці.

3. Робота напряму через індекс (Heap Fetches: 0)

База даних виявилася настільки «розумною», що їй навіть не довелося зазирати в основну таблицю клієнтів (Heap). Усі необхідні дані (ID клієнта) вона взяла безпосередньо із самої структури індексу (Index Only Scan). Це заощадило купу часу та повністю прибрало зайві рухи.


# Порівняльний аналіз: CTE vs. Materialized View:
\begin{array}{|c|c|c|c|}
\hline
\text
Метрика & Сирий\ CTE-пайплайн & Materialized\ View & Ефект\ оптимізації \\
\hline
Execution\ Time & 4.398\ с & 0.091\ мс & Прискорення\ у\ ~48 000\ разів! \\
Buffers\ (Shared Hit) & 111 255\ блоків\ (870\ МБ) & 8\ блоков\ (64\ КБ) &
Зменшення\ навантаження\ на\ диск(RAM)\ у\ 14 000\ разів  \\
Temp\ Disk\ (Read/Write) & 73\ МБ\ (temp\ read=9185) & 0\ МБ\ (немає\ скидання\ на\ диск) & Повне\ уникнення\ дискового\ I/O\ overhead\\
Основний\ оператор & External\ Merge\ Disk\ Sort,\ Hash\ Join & Index\ Scan,\ Nested\ Loop & Перехід\ від\ важкого\ CPU-аналізу\ до\ точкового\ пошуку \\
\hline
\end{array}


Top-N читання з MV:

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
17 rows affected.


QUERY PLAN
Limit (cost=4449.01..4450.15 rows=10 width=92) (actual time=27.376..28.442 rows=10.00 loops=1)
Buffers: shared hit=1637
-> Gather Merge (cost=4449.01..15782.48 rows=99442 width=92) (actual time=27.374..28.436 rows=10.00 loops=1)
Workers Planned: 1
Workers Launched: 1
Buffers: shared hit=1637
-> Sort (cost=3449.00..3595.24 rows=58495 width=92) (actual time=24.376..24.378 rows=8.50 loops=2)
Sort Key: monetary DESC NULLS LAST
Sort Method: top-N heapsort Memory: 26kB
Buffers: shared hit=1637



# Reflection
1. Який feature з pipeline найдорожчий у обчисленні за EXPLAIN ANALYZE? Чому: JOIN, GROUP BY, window function, Sort або Hash Aggregate?    

Аналіз плану виконання EXPLAIN ANALYZE для початкового CTE-пайплайну показує, що найдорожчими операціями є віконні функції (WindowAgg) у поєднанні з багатоетапними сортуваннями (Sort) та Hash Aggregate. Обчислення ковзного 30-денного середнього чека (stage4_rolling_aov) вимагає побудови вікна з сортуванням за order_ts для кожного клієнта. Оскільки базові таблиці olist_orders та olist_order_items містять понад 100 тисяч рядків, операції сортування виходять за межі work_mem на диск (Spill to Disk), а Hash Join для визначення категорій (stage6) створює колосальне навантаження на CPU та пам'ять.    

2. Чи варто кешувати цей feature через Materialized View? Який refresh-cycle ви б обрали: щогодини, щодоби, on-demand?    

Кешування такого важкого пайплайну через Materialized View (MV) є повністю виправданим. Для розрахунку RFM-метрик та когорт історичного датасету Olist оптимальним вибором є щодобовий (daily) розклад оновлення у зручний час або оновлення on-demand після завантаження батчу нових даних. Оскільки аналітика поведінки клієнтів не вимагає мікросекундної актуальності, оновлювати MV щогодини немає сенсу — це б створювало надлишкове навантаження на систему через постійні важкі переобчислення.    

3. У якому use case звичайний VIEW був би правильнішим вибором, ніж MV?   

Звичайний VIEW був би кращим вибором у сценаріях, де критично важлива актуальність даних у реальному часі (real-time freshness). Наприклад, якщо операційному менеджеру на дашборді потрібно бачити скасування замовлень (cancellation_rate) в ту ж секунду, коли клієнт натиснув кнопку скасування, кешовані дані MV відображали б застарілу картину. Також звичайний View ефективніший, коли запити завжди йдуть із точковою фільтрацією за конкретним customer_id і база може використати індекси базових таблиць без повного сканування пайплайну.    

4. Як recursive CTE забезпечує termination? Що станеться, якщо referral graph містить цикл A → B → A?   

Щодо recursive CTE, термінація (termination) забезпечується тим, що рекурсивна частина повертає порожню множину рядочків (коли у наступній ітерації JOIN не знаходить нових збігів). Якщо реферальний граф містить цикл A → B → A без додаткових запобіжників рекурсія перетвориться на нескінченний цикл, що призведе до переповнення пам'яті або помилки таймауту. Для захисту від циклів використовують масиви для відстеження пройдених вузлів (WHERE A.id != ALL(visited_array)) або лічильник глибини рекурсії.   

5. Який trade-off важливіший у цьому pipeline: freshness, latency, recompute-cost чи storage-cost?    

Latency (Швидкість) — це пріоритет №1 Попередній сирий CTE-пайплайн виконувався майже 4,4 секунди. Для "живих" запитів (наприклад, відображення профілю клієнта в адмінці або особистому кабінеті) це було дуже повільно. Перехід на Materialized View (MV) дав прискорення до 0.091 мс. Оскільки метою оптимізації було перетворення "важкого CPU-аналізу на точечний пошук", швидкість тут є головною бізнес-вимогою.

Materialized View «заморожує» дані. Це означає, що якщо клієнт зробить покупку секунду тому, MV не покаже її, поки  матеріалізоване представлення (REFRESH MATERIALIZED VIEW) не оновиться. Розрахунок аналітики — як-от RFM-аналіз, когортний аналіз та історія скасувань — це стратегічні метрики. Маркетологам або аналітикам зазвичай не потрібно знати когорту клієнта з точністю до секунди. Дані, що оновлюються раз на годину або навіть раз на добу, повністю задовольняють бізнес-потреби.

Вартість зберігання (Storage-cost) стала доступною. Раніше при кожному кліку користувача база даних створювала на диску ~73 МБ важких тимчасових файлів, тому що їй не вистачало оперативної пам'яті. Тепер зберігається компактна, готова таблиця, яка займає мінімум місця і повністю звільняє жорсткий диск від постійної та шкідливої біготни (навантаження на I/O).
Витрати на перерахунок (Recompute-cost) стали передбачуваними. Під час кожного візиту клієнта, ця робота перейшла у фон. Тепер база спокійно рахує все за розкладом. Користувачі отримують миттєву відповідь, а процесор сервера має мінімальне навантаження.
